In [1]:
# Install necessary libraries if not already installed
!pip install nltk
import pandas as pd
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Download necessary NLTK data
nltk.download('stopwords')
nltk.download('wordnet')

# Define the URL for the dataset
data_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip'
zip_file_path = 'smsspamcollection.zip'
extracted_file_path = 'SMSSpamCollection'

# Download the dataset
import requests
import zipfile
import os

if not os.path.exists(extracted_file_path):
    print(f"Downloading dataset from {data_url}...")
    response = requests.get(data_url)
    with open(zip_file_path, 'wb') as f:
        f.write(response.content)
    print("Download complete. Extracting...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Extraction complete.")
    os.remove(zip_file_path)
else:
    print("Dataset already exists.")

# Load the dataset
df = pd.read_csv(extracted_file_path, sep='\t', header=None, names=['label', 'message'])

display(df.head())
print(f"Dataset shape: {df.shape}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Download complete. Extracting...
Extraction complete.


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Dataset shape: (5572, 2)


Now that we have loaded the data, let's preprocess the text messages. This involves converting labels, cleaning the text (removing punctuation, lowering case), and tokenizing it.

In [4]:
# Install Flask and joblib if not already installed
!pip install Flask joblib nltk

from flask import Flask, request, jsonify, render_template
import joblib
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# Download necessary NLTK data (if not already downloaded in the environment where Flask runs)
try:
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError: # Corrected from nltk.downloader.DownloadError
    nltk.download('stopwords')
    nltk.download('wordnet')

app = Flask(__name__)

# Load the pre-trained model and TF-IDF vectorizer
tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')
model = joblib.load('multinomial_nb_model.pkl')

# Initialize WordNetLemmatizer and stopwords (must be the same as used during training)
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Remove non-alphabetic characters and convert to lowercase
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    # Tokenize and remove stopwords, then lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

@app.route('/')
def home():
    return render_template('index.html')

@app.route('/predict', methods=['POST'])
def predict():
    if request.method == 'POST':
        message = request.form['message']
        processed_message = preprocess_text(message)
        vectorized_message = tfidf_vectorizer.transform([processed_message])
        prediction = model.predict(vectorized_message)[0]

        output = "Spam" if prediction == 1 else "Ham"

        return render_template('index.html', prediction_text=f'The message is: {output}')

# To run the app, you would typically use `app.run(debug=True)`
# However, for Colab, we'll need to expose it publicly via ngrok.
# We will do this in the next step.

ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information about how to avoid this problem.


ERROR:root:Unexpected exception finding object shape
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/google/colab/_debugpy_repr.py", line 54, in get_shape
    shape = getattr(obj, 'shape', None)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 318, in __get__
    obj = instance._get_current_object()
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/werkzeug/local.py", line 519, in _get_current_object
    raise RuntimeError(unbound_message) from None
RuntimeError: Working outside of request context.

This typically means that you attempted to use functionality that needed
an active HTTP request. Consult the documentation on testing for
information about how to avoid this problem.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_d

FileNotFoundError: [Errno 2] No such file or directory: 'tfidf_vectorizer.pkl'

For the Flask application to work, we need an `index.html` file in a `templates` directory. This file will serve as our web interface. Let's create this directory and file.

In [11]:
# Install pyngrok to expose the Flask app publicly
!pip install pyngrok

from pyngrok import ngrok
import threading
import time
from google.colab import userdata

# Essential imports for Flask app (moved here for robustness)
from flask import Flask, request, jsonify, render_template
import joblib
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# Download necessary NLTK data (if not already downloaded in the environment where Flask runs)
# This helps ensure consistency within the thread's environment
try:
    nltk.data.find('corpora/stopwords')
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('stopwords')
    nltk.download('wordnet')

# Retrieve ngrok auth token from Colab Secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Run Flask in a separate thread so it doesn't block Colab execution
def run_flask_app():
    # Define Flask app and its components inside the thread function
    # This ensures 'app' is defined within the thread's scope and has access to its dependencies
    app = Flask(__name__)

    # Load the pre-trained model and TF-IDF vectorizer
    # These files are assumed to be created by previous cells (9e8db68c)
    tfidf_vectorizer = joblib.load('tfidf_vectorizer.pkl')
    model = joblib.load('multinomial_nb_model.pkl')

    # Initialize WordNetLemmatizer and stopwords (must be the same as used during training)
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    # Preprocessing function (copied from 0ad8bbd8)
    def preprocess_text(text):
        # Remove non-alphabetic characters and convert to lowercase
        text = re.sub('[^a-zA-Z]', ' ', text)
        text = text.lower()
        # Tokenize and remove stopwords, then lemmatize
        words = text.split()
        words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
        return ' '.join(words)

    @app.route('/')
    def home():
        return render_template('index.html')

    @app.route('/predict', methods=['POST'])
    def predict():
        if request.method == 'POST':
            message = request.form['message']
            processed_message = preprocess_text(message)
            vectorized_message = tfidf_vectorizer.transform([processed_message])
            prediction = model.predict(vectorized_message)[0]

            output = "Spam" if prediction == 1 else "Ham"

            return render_template('index.html', prediction_text=f'The message is: {output}')

    # Run the Flask app without debug/reloader in a threaded environment
    app.run(port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask_app)
flask_thread.daemon = True # Allow the main program to exit even if the thread is still running
flask_thread.start()

time.sleep(5) # Give Flask a moment to start and ensure files are loaded

# Set ngrok authtoken and open a tunnel
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000)

print(f"* Flask app running on port 5000")
print(f"* Public URL: {public_url}")
print("To stop the Flask app and ngrok tunnel, interrupt this cell's execution.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


* Flask app running on port 5000
* Public URL: NgrokTunnel: "https://remission-unfounded-demanding.ngrok-free.dev" -> "http://localhost:5000"
To stop the Flask app and ngrok tunnel, interrupt this cell's execution.


In [12]:
ham_messages = df[df['label'] == 0]
display(ham_messages[['message']])
print(f"Total ham messages found: {len(ham_messages)}")

,message
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...
3,U dun say so early hor... U c already then say...
4,"Nah I don't think he goes to usf, he lives aro..."
6,Even my brother is not like to speak with me. ...
...,...
5565,Huh y lei...
5568,Will ü b going to esplanade fr home?
5569,"Pity, * was in mood for that. So...any other s..."
5570,The guy did some bitching but I acted like i'd...


Total ham messages found: 4825


In [13]:
spam_messages = df[df['label'] == 1]
display(spam_messages[['message']])
print(f"Total spam messages found: {len(spam_messages)}")

,message
2,Free entry in 2 a wkly comp to win FA Cup fina...
5,FreeMsg Hey there darling it's been 3 week's n...
8,WINNER!! As a valued network customer you have...
9,Had your mobile 11 months or more? U R entitle...
11,"SIX chances to win CASH! From 100 to 20,000 po..."
...,...
5537,Want explicit SEX in 30 secs? Ring 02073162414...
5540,ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547,Had your contract mobile 11 Mnths? Latest Moto...
5566,REMINDER FROM O2: To get 2.50 pounds free call...


Total spam messages found: 747


In [ ]:
spam_messages = df[df['label'] == 1]
display(spam_messages[['message']])
print(f"Total spam messages found: {len(spam_messages)}")

4.  **Run the Cell:**
    *   Once you've updated the cell as shown above and added your `ngrok` token to Colab secrets, execute this code cell.
    *   It will install `pyngrok`, start your Flask app in the background, and then connect `ngrok`.

5.  **Get Your Public URL:**
    *   After successful execution, the output of the cell will display a line like `* Public URL: https://<some-random-id>.ngrok.io`. This is your public link!
    *   You can open this URL in any web browser to interact with your deployed Flask application.

### Important Notes:
*   The public URL generated by `ngrok` changes every time you run the `ngrok.connect()` command (unless you have a paid `ngrok` plan and configure a fixed domain).
*   Your Flask app will only be accessible as long as this Colab notebook session is active and the `ngrok` cell is running. If you close the notebook or the cell's execution stops, the link will become inactive.
*   To stop the Flask app and the `ngrok` tunnel, simply interrupt the execution of the cell where `ngrok.connect()` was called (the cell you just ran).

In [10]:
import os

# Create the templates directory if it doesn't exist
if not os.path.exists('templates'):
    os.makedirs('templates')

# Write the HTML content to index.html inside the templates directory
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Spam/Ham Detector</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 40px; background-color: #f4f4f4; }
        .container { background-color: #ffffff; padding: 20px; border-radius: 8px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); max-width: 600px; margin: auto; }
        h1 { color: #333; text-align: center; }
        form { display: flex; flex-direction: column; }
        textarea { padding: 10px; margin-bottom: 20px; border: 1px solid #ddd; border-radius: 4px; height: 100px; font-size: 16px; }
        input[type=\"submit\"] { background-color: #4CAF50; color: white; padding: 10px 15px; border: none; border-radius: 4px; cursor: pointer; font-size: 16px; }
        input[type=\"submit\"]:hover { background-color: #45a049; }
        .prediction-result { margin-top: 20px; padding: 15px; border-radius: 4px; background-color: #e9e9e9; text-align: center; font-size: 18px; font-weight: bold; }
    </style>
</head>
<body>
    <div class=\"container\">
        <h1>Spam/Ham Detector</h1>
        <form action=\"/predict\" method=\"POST\">
            <label for=\"message\">Enter your message:</label>
            <textarea name=\"message\" id=\"message\" placeholder=\"Type your message here...\"></textarea>
            <input type=\"submit\" value=\"Predict\">
        </form>
        {% if prediction_text %}
            <div class=\"prediction-result\">{{ prediction_text }}</div>
        {% endif %}
    </div>
</body>
</html>
"""

with open('templates/index.html', 'w') as f:
    f.write(html_content)

print("Created templates/index.html")

Created templates/index.html


In [9]:
import joblib

# Save the TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')
print("TF-IDF Vectorizer saved as tfidf_vectorizer.pkl")

# Save the trained model
joblib.dump(model, 'multinomial_nb_model.pkl')
print("Multinomial Naive Bayes model saved as multinomial_nb_model.pkl")

TF-IDF Vectorizer saved as tfidf_vectorizer.pkl
Multinomial Naive Bayes model saved as multinomial_nb_model.pkl


In [7]:
# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000) # Limiting to 5000 features for practical reasons

# Fit and transform the processed messages
X = tfidf_vectorizer.fit_transform(df['processed_message']).toarray()
y = df['label']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (4457, 5000)
Shape of X_test: (1115, 5000)
Shape of y_train: (4457,)
Shape of y_test: (1115,)


In [8]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Initialize and train the Multinomial Naive Bayes model
model = MultinomialNB()
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.4f}")
print(f"Model Precision: {precision:.4f}")
print(f"Model Recall: {recall:.4f}")
print(f"Model F1-Score: {f1:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)

Model Accuracy: 0.9722
Model Precision: 0.9917
Model Recall: 0.7987
Model F1-Score: 0.8848

Confusion Matrix:
[[965   1]
 [ 30 119]]


In [6]:
# Convert labels to numerical (spam=1, ham=0)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# Initialize WordNetLemmatizer and stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Remove non-alphabetic characters and convert to lowercase
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    # Tokenize and remove stopwords, then lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

df['processed_message'] = df['message'].apply(preprocess_text)

display(df.head())
print(f"Number of processed messages: {len(df['processed_message'].iloc[0])}")

,label,message,processed_message
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,0,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,0,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think go usf life around though


Number of processed messages: 82
